# Notebook 2: Serial communication with the Keithley 2000

*Python for Instrumentation and Laboratory Experiments, Pr. T. Angot*

The Keithley 2000 is connected through an RS232 to USB adapter. Port settings: 9600 baud, 8 data bits, no parity, 1 stop bit. Commands are SCPI text ending with `\r`.

Import the serial library, scan for port to identify the serial com, define the com address

In [1]:
import serial

In [2]:
import serial.tools.list_ports as port_list
ports = list(port_list.comports())
for port in ports: print(port)

COM3 - USB Serial Port (COM3)


In [3]:
serial_address="COM3"
print(serial_address)

COM3


## Preliminary work to test the serial communication with the Keithley2000

Ask for the instrument ID, set the insturment to Volt DC, ask that the instrument to check if it is correctly configured in this mode configured, set the integration time to 5, the range to mV <br>
Retrieve one data

In [4]:
with serial.Serial(serial_address,baudrate=9600,bytesize=8,parity='N',stopbits=1,timeout=.05) as ser:

    ser.write(b'*IDN?\r')
    l=ser.readline()
    response=l.decode('ascii',errors='replace')
    print(response)


KEITHLEY INSTRUMENTS INC.,MODEL 2000,4416105,B01  /A02  



In [5]:
with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=1) as ser:
    ser.write(b":CONF:VOLT:DC\r")
    ser.write(b":SENS:FUNC?\r")
    print(ser.readline().decode('ascii').strip())   # should print "VOLT:DC"

"VOLT:DC"


NPLC 5: integration over 5 power line cycles (100 ms), which averages out the 50 Hz noise. Range 0.1: the 100 mV range, adapted to a thermocouple.

In [6]:
import time

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=2) as ser:
    ser.write(b":SENS:FUNC 'VOLT:DC'\r")
    ser.write(b":SENS:VOLT:DC:NPLC 5\r")
    ser.write(b":SENS:VOLT:DC:RANG 0.1\r")
    time.sleep(0.5)

    for cmd in [b":SENS:FUNC?\r", b":SENS:VOLT:DC:NPLC?\r", b":SENS:VOLT:DC:RANG?\r"]:
        ser.reset_input_buffer()
        ser.write(cmd)
        print(cmd.decode().strip(), "->", ser.readline().decode('ascii').strip())

:SENS:FUNC? -> "VOLT:DC"
:SENS:VOLT:DC:NPLC? -> 5.00
:SENS:VOLT:DC:RANG? -> 0.100000


In [7]:
import time

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=3) as ser:
    ser.reset_input_buffer()
    ser.write(b":READ?\r")
    reading = ser.readline().decode('ascii').strip()
    print("Reading:", reading, "V")

Reading: +3.04687512E-02 V


Transform the data to a float number

In [8]:
value = float(reading)
print(value, "V")
print(value * 1000, "mV")

0.0304687512 V
30.4687512 mV


No thermocouple was connected to the input during the session, so this value is just the floating open input and has no physical meaning.